In [1]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [4]:
device = torch.device("cuda")
device

device(type='cuda')

In [5]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small", torch_dtype=torch.float32)
tokenizer.pad_token = tokenizer.eos_token
tokenizer

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


GPT2Tokenizer(name_or_path='microsoft/DialoGPT-small', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [6]:
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-small",torch_dtype=torch.float32)
model

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [7]:
model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [8]:
dataset = pd.read_csv("normalized_context_and_response.csv")
dataset.head()

,context,response
0,i'm going through some things with my feelings...,if everyone thinks you're worthless then maybe...
1,i'm going through some things with my feelings...,hello and thank you for your question and seek...
2,i'm going through some things with my feelings...,first thing i'd suggest is getting the sleep y...
3,i'm going through some things with my feelings...,therapy is essential for those that are feelin...
4,i'm going through some things with my feelings...,i first want to let you know that you are not ...


In [9]:
contexts = dataset['context'].astype('str').values
contexts[:5]

array(["i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthless to everyone",
       "i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthless to everyone",
       "i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthle

In [10]:
responses = dataset['response'].astype('str').values
responses[:5]

array(["if everyone thinks you're worthless then maybe you need to find new people to hang out withseriously the social context in which a person lives is a big influence in self-esteemotherwise you can go round and round trying to understand why you're not worthless then go back to the same crowd and be knocked down againthere are many inspirational messages you can find in social media \xa0maybe read some of the ones which state that no person is worthless and that everyone has a good purpose to their lifealso since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terriblebad feelings are part of living \xa0they are the motivation to remove ourselves from situations and relationships which do us more harm than goodbad feelings do feel terrible \xa0 your feeling of worthlessness may be good in the sense of motivating you to find out that you are much better than your feelings today",
       "hello and thank you for you

In [11]:
def combineText(example):
    return {
        "text": "User: " + example['context'] +
            "Bot: " + example['response']
    }

In [12]:
def encode(example):
    return tokenizer(example['text'], truncation=True, padding=True, max_length=64)

In [13]:
def add_labels(example):
    example['labels']=example['input_ids']
    return example

In [14]:
datasets = Dataset.from_dict({
    "context": contexts,
    "response": responses
})
datasets

Dataset({
    features: ['context', 'response'],
    num_rows: 3512
})

In [15]:
datasets_split = datasets.train_test_split(test_size=0.2)
datasets_split

DatasetDict({
    train: Dataset({
        features: ['context', 'response'],
        num_rows: 2809
    })
    test: Dataset({
        features: ['context', 'response'],
        num_rows: 703
    })
})

In [16]:
trainSet = datasets_split['train']
trainSet

Dataset({
    features: ['context', 'response'],
    num_rows: 2809
})

In [17]:
testSet = datasets_split['test']
testSet

Dataset({
    features: ['context', 'response'],
    num_rows: 703
})

In [18]:
trainSet = trainSet.map(combineText)
trainSet

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text'],
    num_rows: 2809
})

In [19]:
testSet = testSet.map(combineText)
testSet

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text'],
    num_rows: 703
})

In [20]:
trainSet = trainSet.map(encode, batched=True)
trainSet

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask'],
    num_rows: 2809
})

In [21]:
testSet = testSet.map(encode, batched=True)
testSet

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask'],
    num_rows: 703
})

In [22]:
trainSet = trainSet.map(add_labels)
trainSet

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2809
})

In [23]:
testSet = testSet.map(add_labels)
testSet

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 703
})

In [24]:
print(set(trainSet[0]["labels"]))

{4096, 257, 644, 389, 262, 1545, 651, 780, 1037, 2962, 661, 20630, 3607, 534, 25, 2842, 1561, 284, 546, 290, 5419, 428, 12982, 318, 21951, 832, 319, 326, 329, 460, 1997, 6476, 3404, 466, 1107, 340, 2130, 1365, 857, 345, 3294, 2272, 612, 749, 621, 761}


In [25]:
trainingArgs = TrainingArguments(
    output_dir="./output",
    num_train_epochs=1,
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    eval_strategy='epoch',
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir=None,
    fp16=False,
    bf16=False,
    max_grad_norm = 1.0
)

In [26]:
trainSet = trainSet.select(range(1100))

In [27]:
testSet = testSet.select(range(670))

In [28]:
trainer=Trainer(
    model=model,
    args=trainingArgs,
    train_dataset= trainSet,
    eval_dataset=testSet
)

In [29]:
trainer.evaluate(testSet)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'eval_loss': 6.313945293426514,
 'eval_model_preparation_time': 0.0114,
 'eval_runtime': 17.1284,
 'eval_samples_per_second': 39.116,
 'eval_steps_per_second': 39.116}

In [31]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,4.473899,3.534694,0.011400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=550, training_loss=4.42277870871804, metrics={'train_runtime': 116.0183, 'train_samples_per_second': 9.481, 'train_steps_per_second': 4.741, 'total_flos': 52231186022400.0, 'train_loss': 4.42277870871804, 'epoch': 1.0})

In [32]:
trainer.save_model("mental-health-dialogpt")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [34]:
trainer.evaluate(testSet)

{'eval_loss': 3.534694194793701,
 'eval_model_preparation_time': 0.0114,
 'eval_runtime': 20.8084,
 'eval_samples_per_second': 32.199,
 'eval_steps_per_second': 32.199,
 'epoch': 1.0}